# Experiment 7: Uber Fare Prediction
Preprocess the dataset, remove outliers, check correlation, and compare Linear Regression with Random Forest.

In [ ]:
import math
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

CSV_FILE = "uber.csv"  # Change this to your downloaded dataset path
df = pd.read_csv(CSV_FILE)
df.head()

In [ ]:
def distance(row):
    lat1, lon1 = math.radians(row.pickup_latitude), math.radians(row.pickup_longitude)
    lat2, lon2 = math.radians(row.dropoff_latitude), math.radians(row.dropoff_longitude)
    a = math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2
    return 6371 * 2 * math.asin(math.sqrt(a))

df = df.dropna()
df = df[df.fare_amount > 0]
df["distance"] = df.apply(distance, axis=1)
low, high = df.fare_amount.quantile([.01, .99])
df = df[df.fare_amount.between(low, high)]
df[["distance", "fare_amount"]].corr()

In [ ]:
X = df[["distance"]]
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
models = {"Linear Regression": LinearRegression(), "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42)}
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    print(name, "R2 =", round(r2_score(y_test, prediction), 3), "RMSE =", round(mean_squared_error(y_test, prediction)**.5, 3))